In [ ]:
import os
from pathlib import Path

# Localiza a raiz do projeto de forma robusta (funciona em VS Code, JupyterLab, etc.)
def _find_project_root():
    # VS Code expõe o caminho do notebook nesta variável
    nb_file = globals().get('__vsc_ipynb_file__') or locals().get('__vsc_ipynb_file__')
    if nb_file:
        return Path(nb_file).resolve().parent.parent
    # JupyterLab / linha de comandos
    try:
        import ipynbname
        return ipynbname.path().parent.parent
    except Exception:
        pass
    # Último recurso: working directory atual sobe um nível
    cwd = Path().resolve()
    if cwd.name == 'notebooks':
        return cwd.parent
    return cwd

PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
print(f"✓ Working directory: {PROJECT_ROOT}")


## Síntese Metodológica: Roteamento Espacial e Restrições Físicas

Este *notebook* consolida a infraestrutura de dados espaciais do projeto, conectando o potencial de geração fotovoltaica dos edifícios à topologia da rede elétrica (E-Redes). Esta versão substitui os ficheiros de entrada por dados corrigidos (`producao_pv_cp7.gpkg` e `VoronoiPTD_Areas_Servico_Rede.gpkg`), que apresentavam inconsistências geométricas na versão anterior. O fluxo de trabalho desenvolvido em grupo baseou-se nos seguintes pilares:

1. **Voronoi de Rede (Roteamento Realista):** A área de serviço de cada Posto de Transformação de Distribuição (PTD) foi calculada utilizando a rede real de ruas (distâncias de trajeto via *OSMnx*), superando a imprecisão das distâncias em linha reta (Voronoi Euclidiano).
2. **Indivisibilidade da Infraestrutura (Centroides):** Para evitar que um mesmo edifício fosse fatiado entre dois PTDs nas zonas de fronteira, reduzimos as geometrias a centroides durante o *Spatial Join*, garantindo uma alocação estrita de cardinalidade 1:1.
3. **Cobertura Híbrida do Mapeamento Espacial:** A tesselação de Voronoi não cobre o território de forma perfeitamente contígua — restam margens de poucos metros entre áreas de serviço adjacentes. O cruzamento espacial combina, por isso, uma união direta (*within*) com um *fallback* por proximidade (`sjoin_nearest`), evitando que falhas residuais de poucos metros eliminem edifícios fisicamente válidos do modelo.
4. **Resolução do Problema de Empacotamento:** Implementou-se uma trava matemática rígida (Fator de Empacotamento de 0.85 + Função *Floor*) para garantir que a estimativa de painéis instaláveis reflete apenas unidades inteiras que cabem fisicamente nas coberturas, descontando áreas ociosas e de manutenção.

# 1. Mapeamento Topológico (Edifícios vs. Áreas de Serviço PTD)

A atribuição do potencial fotovoltaico à infraestrutura da rede de distribuição requer o cruzamento espacial entre as pegadas dos edifícios e os polígonos de Voronoi gerados para os Postos de Transformação de Distribuição (PTD).

Metodologicamente, optou-se por realizar a união espacial (*Spatial Join*) através do **centroide geométrico** de cada edifício, em detrimento da interseção total de polígonos. Esta decisão teórica sustenta-se em dois eixos:

1. **Garantia de Indivisibilidade:** A infraestrutura predial atua como um nó único de injeção/consumo na rede elétrica. A utilização de polígonos geraria falsas duplicações ou fragmentações irrealistas em edifícios localizados nas linhas de fronteira de duas áreas de Voronoi. O centroide assegura uma alocação 1:1 unívoca.
2. **Proxy de Ligação:** Na ausência de microdados sobre o traçado exato dos ramais de ligação de baixa tensão para cada fração, o centro de massa do polígono assume a função de representação espacial (ponto ótimo) da carga térmica e do potencial de geração daquela geometria.

A transformação é estritamente topológica para efeitos de roteamento, mantendo-se inalterados todos os atributos escalares de área útil e potência instalável calculados nas fases anteriores.

**Nota sobre cobertura espacial (auditoria dos dados corrigidos):** apenas 332 dos 936 edifícios (35,5%) têm o centroide estritamente contido nalgum polígono de Voronoi. Isto não decorre de um erro de projeção ou alinhamento de CRS, mas do facto de a tesselação de rede deixar pequenas margens entre áreas de serviço adjacentes (mediana de ~15 m de distância ao polígono mais próximo; 75% dos casos a menos de 87 m). Excluir esses ~600 edifícios por uma questão de poucos metros de fronteira distorceria gravemente o resultado — descartaria cerca de 70% da potência fotovoltaica teórica do município. O mapeamento foi por isso ajustado para combinar:

1. **Match direto** (`predicate="within"`) sempre que o centroide cai estritamente dentro de um polígono;
2. **Match por proximidade** (`sjoin_nearest`, limite de 1.000 m) para os restantes, atribuindo o edifício ao PTD de serviço mais próximo.

Apenas 3 edifícios (CP7 3800-901, 3800-902 e 3800-903) ficaram a mais de 1 km de qualquer área de serviço e foram excluídos do modelo — os códigos terminados em "9xx" sugerem tratar-se de códigos postais especiais/institucionais, mas isto não foi confirmado de forma independente.

In [3]:
import geopandas as gpd
import pandas as pd
import warnings
warnings.filterwarnings('ignore') # Oculta avisos do cálculo de centroides

print("1. A carregar os ficheiros...")
gdf_edificios = gpd.read_file(r"data/raw/producao_pv_cp7.gpkg")
gdf_ptd = gpd.read_file(r"data/raw/VoronoiPTD_Areas_Servico_Rede.gpkg")

# Alinhamento de Sistemas de Coordenadas
if gdf_edificios.crs != gdf_ptd.crs:
    print("2. A alinhar as coordenadas geográficas...")
    gdf_ptd = gdf_ptd.to_crs(gdf_edificios.crs)

print("3. A otimizar geometrias (transformando edifícios em pontos)...")
gdf_pontos = gdf_edificios.copy()
gdf_pontos["geometry"] = gdf_edificios.geometry.centroid
gdf_pontos["orig_idx"] = gdf_pontos.index

print("4. A cruzar os mapas — match direto (within)...")
# Passo 1: união espacial direta (centroide estritamente dentro do polígono)
direto = gpd.sjoin(gdf_pontos, gdf_ptd[['ptd_id', 'geometry']], how="inner", predicate="within")
direto = direto.drop_duplicates(subset="orig_idx", keep="first")  # edifícios exatamente na fronteira de 2 polígonos
direto["tipo_match"] = "direto"
direto["dist_m"] = 0.0

print("5. A resolver edifícios fora de qualquer polígono — fallback por proximidade...")
# Passo 2: a tesselação de Voronoi deixa pequenas margens entre áreas de serviço adjacentes;
# para não perder edifícios válidos por uma questão de poucos metros de fronteira, aplicamos
# um fallback por distância mínima (sjoin_nearest) aos que não caíram dentro de nenhum polígono.
LIMITE_DISTANCIA_M = 1000  # acima disto, consideramos o edifício fora da rede modelada

nao_mapeados = gdf_pontos[~gdf_pontos["orig_idx"].isin(direto["orig_idx"])]
fallback = gpd.sjoin_nearest(nao_mapeados, gdf_ptd[['ptd_id', 'geometry']], how="left", distance_col="dist_m")
fallback = fallback.sort_values("dist_m").drop_duplicates(subset="orig_idx", keep="first")

fallback_ok = fallback[fallback["dist_m"] <= LIMITE_DISTANCIA_M].copy()
fallback_excluidos = fallback[fallback["dist_m"] > LIMITE_DISTANCIA_M].copy()
fallback_ok["tipo_match"] = "nearest_fallback"

gdf_mapeado = pd.concat([direto, fallback_ok], ignore_index=True)

print(f"\n=== DIAGNÓSTICO DO MAPEAMENTO ===")
print(f"Total de edifícios de entrada:                {len(gdf_edificios)}")
print(f"  Match direto (within):                      {len(direto)}")
print(f"  Match por proximidade (fallback ≤{LIMITE_DISTANCIA_M}m):     {len(fallback_ok)}")
print(f"  Excluídos (>{LIMITE_DISTANCIA_M}m de qualquer PTD):          {len(fallback_excluidos)}")
print(f"  TOTAL mapeado:                               {len(gdf_mapeado)} ({len(gdf_mapeado)/len(gdf_edificios)*100:.1f}%)")
assert len(gdf_mapeado) + len(fallback_excluidos) == len(gdf_edificios), "A contagem de edifícios não fecha!"

if len(fallback_excluidos) > 0:
    print("\nEdifícios excluídos do modelo (fora do alcance da rede modelada):")
    print(fallback_excluidos[["cp7", "area_util_m2", "potencia_kwp", "dist_m"]].to_string(index=False))

# Converte para tabela e guarda
df_final = pd.DataFrame(gdf_mapeado.drop(columns=['geometry', 'orig_idx', 'index_right']))
ficheiro_saida = r"data/processed/potencial_com_mapeamento_ptd.csv"
df_final.to_csv(ficheiro_saida, index=False, sep=";", encoding="utf-8-sig")

print(f"\n✓ Cruzamento concluído com sucesso!")
print(f"Ficheiro '{ficheiro_saida}' guardado com {len(df_final)} registos.")

1. A carregar os ficheiros...
3. A otimizar geometrias (transformando edifícios em pontos)...
4. A cruzar os mapas — match direto (within)...
5. A resolver edifícios fora de qualquer polígono — fallback por proximidade...

=== DIAGNÓSTICO DO MAPEAMENTO ===
Total de edifícios de entrada:                936
  Match direto (within):                      332
  Match por proximidade (fallback ≤1000m):     601
  Excluídos (>1000m de qualquer PTD):          3
  TOTAL mapeado:                               933 (99.7%)

Edifícios excluídos do modelo (fora do alcance da rede modelada):
     cp7  area_util_m2  potencia_kwp      dist_m
3800-902    214.319789     42.863958 4751.308379
3800-901   5136.754848   1027.350970 4903.131933
3800-903   3287.762192    657.552438 5350.302348

✓ Cruzamento concluído com sucesso!
Ficheiro 'C:\Documentos\UA_Mestrado\1UA\2 semestre\seminario\Milestone IV\numero de paines a se instalar por PTD\02_Dados_Processados\potencial_com_mapeamento_ptd.csv' guardado com 933

## 2. Dimensionamento Físico e Restrições Geométricas (Problema de Empacotamento)

O cálculo do número de painéis fotovoltaicos suportados pela rede de distribuição não pode ser derivado da simples divisão da área total agregada de um Posto de Transformação pela área unitária de um painel. Essa abordagem ignora o **Problema de Empacotamento** (*Packing Problem*), gerando superestimativas irreais, visto que módulos solares são estruturas retangulares rígidas que não preenchem perfeitamente polígonos irregulares (telhados).

Para garantir a viabilidade física e a exequibilidade do modelo, o algoritmo implementa um conjunto de restrições aplicadas de forma descentralizada (**edifício a edifício**) antes de qualquer agregação para a rede:

1. **Fator de Empacotamento (0.85):** Aplica-se um coeficiente de redução que assume que 15% da área útil contínua de um telhado será invariavelmente perdida devido a exigências físicas e normativas (margens de recuo, corredores de manutenção para limpeza, sombreamentos pontuais de chaminés e claraboias).
2. **Restrição de Integridade (Função *Floor*):** Sendo fisicamente impossível instalar frações de um módulo rígido, o modelo aplica a função matemática piso (`np.floor`) para forçar o arredondamento estrito para o número inteiro inferior.
3. **Dupla Validação (Mínimo Viável):** O limite máximo de painéis alocado a cada infraestrutura é determinado pelo valor mínimo entre a restrição de espaço (área) e a restrição elétrica (potência máxima calculada para aquele edifício).

A formulação matemática principal da restrição espacial por infraestrutura obedece à equação:

$$Painéis_{Físico} = \lfloor \frac{\text{Área\_Útil} \times Fator\_Empacotamento}{\text{Área\_Unitária\_do\_Painel}} \rfloor$$

**Nota Metodológica:** Embora a geometria topológica dos edifícios tenha sido reduzida a centroides na etapa de união espacial (*Spatial Join*), o rigor deste cálculo matemático permanece inalterado. O centroide atua unicamente como um vetor de roteamento para o Posto de Transformação de Distribuição (PTD), enquanto os atributos bidimensionais de capacidade (como a `área_útil_m2`) transitaram intactos na estrutura tabular dos dados.

In [4]:
import pandas as pd
import numpy as np

# ==========================================
# 1. PARÂMETROS TÉCNICOS E FÍSICOS
# ==========================================
POTENCIA_PAINEL_KWP = 0.450  # Potência unitária do módulo em kWp (450W)
AREA_PAINEL_M2 = 2.0         # Área física estrita do módulo (m²)

# Fator de Empacotamento: Limita a área para passagens, inclinação e manutenção
FATOR_EMPACOTAMENTO = 0.85 

print("A carregar o mapeamento espacial...")
# 2. CARREGAMENTO DOS DADOS MAPEADOS
caminho_dados = r"data/processed/potencial_com_mapeamento_ptd.csv" 
df_dados = pd.read_csv(caminho_dados, sep=";", encoding="utf-8-sig")
df_dados.columns = df_dados.columns.str.strip()

print("A calcular a viabilidade física edifício a edifício...")
# ==========================================
# 3. CÁLCULO DESCENTRALIZADO (A TRAVA DO PROFESSOR)
# ==========================================
# Trava 1 (Espaço): Aplica fator de empacotamento, divide pela área do painel e arredonda para baixo (floor)
df_dados["paineis_por_area"] = np.floor((df_dados["area_util_m2"] * FATOR_EMPACOTAMENTO) / AREA_PAINEL_M2)

# Trava 2 (Potência): Divide a potência total do telhado pela potência do painel e arredonda para baixo
df_dados["paineis_por_potencia"] = np.floor(df_dados["potencia_kwp"] / POTENCIA_PAINEL_KWP)

# Decisão Realista por Telhado: O menor valor inteiro entre espaço e potência
df_dados["paineis_instalaveis"] = df_dados[["paineis_por_area", "paineis_por_potencia"]].min(axis=1).astype(int)

# Diagnóstico de edifícios inviáveis (demasiado pequenos)
edificios_inviaveis = len(df_dados[df_dados["paineis_instalaveis"] == 0])
print(f"Edifícios rejeitados por falta de espaço físico realista: {edificios_inviaveis}")

print("\nA agregar potencial por PTD...")
# ==========================================
# 4. AGREGAÇÃO TOPOLÓGICA PARA A REDE
# ==========================================
# Agrupamos pelo ID do PTD (agora somando apenas painéis reais e inteiros)
agg_ptd = df_dados.groupby("ptd_id").agg(
    total_paineis_reais=("paineis_instalaveis", "sum"),
    potencia_total_kwp=("potencia_kwp", "sum"),
    area_util_total_m2=("area_util_m2", "sum"),
    n_edificios_viaveis=("paineis_instalaveis", lambda x: (x > 0).sum())
).reset_index()

# ==========================================
# 5. DIAGNÓSTICO FINAL E EXPORTAÇÃO
# ==========================================
print("\n=== RESUMO REALISTA POR PTD ===")
print(f"Total de PTDs avaliados: {len(agg_ptd)}")
print(f"Total de painéis viáveis na rede: {agg_ptd['total_paineis_reais'].sum():,}")

print("\nTop 5 PTDs com maior capacidade de instalação:")
print(agg_ptd.sort_values("total_paineis_reais", ascending=False).head(5).to_string(index=False))

# Guardar os resultados
ficheiro_saida = r"data/processed/dimensionamento_realista_paineis_ptd.csv"
agg_ptd.to_csv(ficheiro_saida, index=False, sep=";", encoding="utf-8-sig")
print(f"\n✓ Resultados guardados com sucesso em: '{ficheiro_saida}'")

A carregar o mapeamento espacial...
A calcular a viabilidade física edifício a edifício...
Edifícios rejeitados por falta de espaço físico realista: 1

A agregar potencial por PTD...

=== RESUMO REALISTA POR PTD ===
Total de PTDs avaliados: 368
Total de painéis viáveis na rede: 412,924

Top 5 PTDs com maior capacidade de instalação:
 ptd_id  total_paineis_reais  potencia_total_kwp  area_util_total_m2  n_edificios_viaveis
    168                 9568         4504.472926        22522.364632                    5
    404                 6735         3169.701195        15848.505973                    1
    470                 6454         3038.927536        15194.637680                    7
    339                 6244         2938.507773        14692.538867                    1
    354                 5995         2821.314960        14106.574802                    1

✓ Resultados guardados com sucesso em: 'C:\Documentos\UA_Mestrado\1UA\2 semestre\seminario\Milestone IV\numero de paines a s

In [5]:
import geopandas as gpd

print("A recalcular métricas energéticas baseadas no dimensionamento físico...")

# ==========================================
# 1. ATUALIZAÇÃO DA POTÊNCIA E PRODUÇÃO REAL
# ==========================================
# Parâmetros resgatados da Fase 1 do vosso projeto
H_ANUAL_PVGIS = 1860.2  # kWh/m²/ano (valor do output do vosso PVGIS)
PR = 0.80               # Performance Ratio

# A Nova Potência Instalada é estritamente o nº de painéis reais * potência unitária
agg_ptd["potencia_real_kwp"] = agg_ptd["total_paineis_reais"] * POTENCIA_PAINEL_KWP

# A Nova Produção usa a fórmula clássica: E = P_kwp * H_anual * PR
# (Lembrando que a eficiência do painel já está embutida no valor em kWp)
agg_ptd["producao_real_kwh_ano"] = agg_ptd["potencia_real_kwp"] * H_ANUAL_PVGIS * PR
agg_ptd["producao_real_mwh_ano"] = agg_ptd["producao_real_kwh_ano"] / 1000

print(f"Total painéis: {agg_ptd['total_paineis_reais'].sum():,}")
print(f"Potência real total: {agg_ptd['potencia_real_kwp'].sum()/1000:.2f} MWp")
print(f"Produção real total: {agg_ptd['producao_real_kwh_ano'].sum()/1e6:.2f} GWh/ano")
print(f"Verificação manual: {412924 * 0.450 * 1860.2 * 0.80 / 1e6:.2f} GWh/ano")

# Limpar as colunas antigas (teóricas) para evitar confusão no Streamlit
agg_ptd = agg_ptd.drop(columns=["potencia_total_kwp"])

print("\n=== RESUMO ENERGÉTICO ATUALIZADO ===")
print(f"Nova Potência Instalável: {agg_ptd['potencia_real_kwp'].sum() / 1000:.2f} MWp")
print(f"Nova Produção Estimada: {agg_ptd['producao_real_mwh_ano'].sum() / 1000:.2f} GWh/ano")

# ==========================================
# 2. EXPORTAÇÃO PARA O MAPA DO STREAMLIT
# ==========================================
print("\nA integrar os dados com os Polígonos de Voronoi (Rede)...")

# Carrega o ficheiro corrigido de áreas de serviço (Voronoi de rede)
gdf_voronoi = gpd.read_file(r"data/raw/VoronoiPTD_Areas_Servico_Rede.gpkg")

# Faz o Join: Colar a nossa tabela de painéis reais e energia na geometria do Voronoi
gdf_final_streamlit = gdf_voronoi.merge(
    agg_ptd, 
    on="ptd_id", 
    how="left"
)

# Preencher PTDs sem painéis com zero, para o mapa não dar erro de dados em falta (NaN)
colunas_preencher = ["total_paineis_reais", "potencia_real_kwp", "producao_real_mwh_ano", "n_edificios_viaveis"]
gdf_final_streamlit[colunas_preencher] = gdf_final_streamlit[colunas_preencher].fillna(0)

# Exportar o ficheiro definitivo para o Dashboard
ficheiro_dashboard = r"data/processed/dashboard_ptd_capacidade_real.gpkg"
gdf_final_streamlit.to_file(ficheiro_dashboard, driver="GPKG")

print(f"\n✓ Processo Concluído! Ficheiro espacial '{ficheiro_dashboard}' gerado com sucesso.")
print("Este ficheiro contém as fronteiras e as métricas exatas prontas para visualização no mapa interativo.")

A recalcular métricas energéticas baseadas no dimensionamento físico...
Total painéis: 412,924
Potência real total: 185.82 MWp
Produção real total: 276.52 GWh/ano
Verificação manual: 276.52 GWh/ano

=== RESUMO ENERGÉTICO ATUALIZADO ===
Nova Potência Instalável: 185.82 MWp
Nova Produção Estimada: 276.52 GWh/ano

A integrar os dados com os Polígonos de Voronoi (Rede)...

✓ Processo Concluído! Ficheiro espacial 'C:\Documentos\UA_Mestrado\1UA\2 semestre\seminario\Milestone IV\numero de paines a se instalar por PTD\02_Dados_Processados\dashboard_ptd_capacidade_real.gpkg' gerado com sucesso.
Este ficheiro contém as fronteiras e as métricas exatas prontas para visualização no mapa interativo.


## 4. Enriquecimento de Dados para Visualização (Streamlit)

Para suportar a aplicação interativa do projeto, a matriz topológica foi enriquecida com inteligência territorial extraída via OpenStreetMap:
* **Toponímia:** Cruzamento espacial com fronteiras administrativas para dotar cada PTD do nome da sua Freguesia.
* **Cenários Interativos:** Identificação de infraestruturas municipais (criando a variável `is_municipal`), permitindo simular o impacto exclusivo do património camarário na formação das CERs.

In [6]:
import osmnx as ox
import geopandas as gpd
import warnings
warnings.filterwarnings('ignore')

print("A iniciar o download de dados territoriais do OpenStreetMap...")
lugar = "Aveiro, Portugal"

# ==========================================
# 1. EXTRAÇÃO DAS FREGUESIAS
# ==========================================
print("\n1. A extrair limites das Freguesias de Aveiro...")
# Em Portugal, o admin_level=8 corresponde às Freguesias
tags_freguesias = {"admin_level": "8"}
gdf_freguesias = ox.features_from_place(lugar, tags_freguesias)

# Filtrar apenas os polígonos (ignorando linhas de fronteira soltas)
gdf_freguesias = gdf_freguesias[gdf_freguesias.geometry.type.isin(['Polygon', 'MultiPolygon'])]
# Manter apenas o nome e a geometria
gdf_freguesias = gdf_freguesias[['name', 'geometry']].rename(columns={'name': 'Freguesia'})

gdf_freguesias.to_file(r"data/processed/freguesias_aveiro.gpkg", driver="GPKG")
print(f"✓ Sucesso: {len(gdf_freguesias)} Freguesias guardadas no ficheiro 'freguesias_aveiro.gpkg'.")

# ==========================================
# 2. EXTRAÇÃO DOS EDIFÍCIOS MUNICIPAIS / PÚBLICOS
# ==========================================
print("\n2. A extrair infraestruturas públicas e camarárias...")
# Procurar edifícios classificados como câmaras municipais, edifícios cívicos ou serviços públicos
tags_publicos = {
    "amenity": ["townhall", "public_building", "community_centre"], 
    "building": ["civic", "public"]
}
gdf_publicos = ox.features_from_place(lugar, tags_publicos)

gdf_publicos = gdf_publicos[gdf_publicos.geometry.type.isin(['Polygon', 'MultiPolygon'])]
gdf_publicos = gdf_publicos[['geometry']] # Precisamos essencialmente da localização

gdf_publicos.to_file(r"data/processed/edificios_publicos_aveiro.gpkg", driver="GPKG")
print(f"✓ Sucesso: {len(gdf_publicos)} edifícios públicos guardados no ficheiro 'edificios_publicos_aveiro.gpkg'.")

A iniciar o download de dados territoriais do OpenStreetMap...

1. A extrair limites das Freguesias de Aveiro...
✓ Sucesso: 22 Freguesias guardadas no ficheiro 'freguesias_aveiro.gpkg'.

2. A extrair infraestruturas públicas e camarárias...
✓ Sucesso: 31 edifícios públicos guardados no ficheiro 'edificios_publicos_aveiro.gpkg'.


In [7]:
import geopandas as gpd
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("=== ENRIQUECIMENTO DE DADOS PARA O STREAMLIT ===")

# ==========================================
# 1. NOMEAÇÃO DAS ZONAS (FREGUESIAS)
# ==========================================
print("\n1. A cruzar os PTDs com as Freguesias de Aveiro...")

# Carrega o mapa final que criámos com as métricas de energia reais
gdf_dashboard = gpd.read_file(r"data/processed/dashboard_ptd_capacidade_real.gpkg")
gdf_freguesias = gpd.read_file(r"data/processed/freguesias_aveiro.gpkg")

# Garante o mesmo sistema de coordenadas
if gdf_dashboard.crs != gdf_freguesias.crs:
    gdf_freguesias = gdf_freguesias.to_crs(gdf_dashboard.crs)

# Usa o centroide do PTD para descobrir a sua Freguesia
gdf_voronoi_pontos = gdf_dashboard.copy()
gdf_voronoi_pontos["geometry"] = gdf_dashboard.geometry.centroid

gdf_ptd_freguesia = gpd.sjoin(
    gdf_voronoi_pontos, 
    gdf_freguesias[['Freguesia', 'geometry']], 
    how="left", 
    predicate="within"
)

# Adiciona o nome da Freguesia ao mapa do Streamlit
gdf_dashboard["zona_freguesia"] = gdf_ptd_freguesia["Freguesia"].fillna("Desconhecida")

# Guarda a versão final absoluta para o mapa
ficheiro_final_mapa = "data/processed/dashboard_ptd_completo.gpkg"
gdf_dashboard.to_file(ficheiro_final_mapa, driver="GPKG")
print(f"✓ Zonas atribuídas com sucesso! Ficheiro '{ficheiro_final_mapa}' gerado.")

# ==========================================
# 2. IDENTIFICAÇÃO DE EDIFÍCIOS MUNICIPAIS / PÚBLICOS
# ==========================================
print("\n2. A identificar edifícios públicos para o Cenário da CER...")

# Carrega os polígonos originais do vosso projeto (CP7)
gdf_cp7 = gpd.read_file(r"data/raw/producao_pv_cp7.gpkg")
gdf_publicos = gpd.read_file(r"data/processed/edificios_publicos_aveiro.gpkg")

if gdf_cp7.crs != gdf_publicos.crs:
    gdf_publicos = gdf_publicos.to_crs(gdf_cp7.crs)

# Descobre quais CP7 intersetam fisicamente os 31 edifícios públicos do OSM
intersecao_publica = gpd.sjoin(
    gdf_cp7, 
    gdf_publicos, 
    how="inner", 
    predicate="intersects"
)

# Cria uma lista com os códigos que possuem infraestrutura pública
lista_cp7_publicos = intersecao_publica['cp7'].unique()

# Atualiza a nossa base de mapeamento com a flag (0 = Privado, 1 = Público)
df_mapeamento = pd.read_csv(r"data/processed/potencial_com_mapeamento_ptd.csv", sep=";", encoding="utf-8-sig")
df_mapeamento['is_municipal'] = df_mapeamento['cp7'].apply(lambda x: 1 if x in lista_cp7_publicos else 0)

ficheiro_csv_cenarios = "data/processed/potencial_com_mapeamento_ptd_enriquecido.csv"
df_mapeamento.to_csv(ficheiro_csv_cenarios, index=False, sep=";", encoding="utf-8-sig")

print(f"✓ Cenário preparado! {df_mapeamento['is_municipal'].sum()} locais identificados como infraestrutura pública.")
print(f"✓ Ficheiro base para cálculos '{ficheiro_csv_cenarios}' atualizado.")
print("\n=== PIPELINE DE DADOS CONCLUÍDO ===")

=== ENRIQUECIMENTO DE DADOS PARA O STREAMLIT ===

1. A cruzar os PTDs com as Freguesias de Aveiro...
✓ Zonas atribuídas com sucesso! Ficheiro 'C:\Documentos\UA_Mestrado\1UA\2 semestre\seminario\Milestone IV\numero de paines a se instalar por PTD\02_Dados_Processados\dashboard_ptd_completo.gpkg' gerado.

2. A identificar edifícios públicos para o Cenário da CER...
✓ Cenário preparado! 12 locais identificados como infraestrutura pública.
✓ Ficheiro base para cálculos 'C:\Documentos\UA_Mestrado\1UA\2 semestre\seminario\Milestone IV\numero de paines a se instalar por PTD\02_Dados_Processados\potencial_com_mapeamento_ptd_enriquecido.csv' atualizado.

=== PIPELINE DE DADOS CONCLUÍDO ===


## 5. Limitações Metodológicas: O Paradigma dos Grandes Consumidores

É necessário registar uma assimetria informacional nos dados do Operador de Rede (E-Redes). O modelo enfrenta lacunas no perfil de Grandes Consumidores (infraestruturas em Média Tensão, como a Universidade de Aveiro e hospitais), omitidos por confidencialidade ou regra de anonimato. 

Para manter o rigor da modelagem sociotécnica, assumem-se as seguintes premissas:
1. **Produtores Âncora:** A área de cobertura destas grandes estruturas foi retida. Elas atuam como produtoras massivas cujo excedente fotovoltaico pode ser injetado na rede local.
2. **Foco na Baixa Tensão:** O âmago de uma CER, sob a ótica da mitigação da pobreza energética e justiça espacial, reside na partilha com o tecido residencial. A ausência de dados de Média Tensão não invalida o modelo; pelo contrário, concentra a análise na demografia que mais beneficia destas estruturas.
3. **Edifícios Excluídos por Cobertura de Rede:** Após a correção dos dados de área de telhado e de áreas de serviço dos PTD, 3 edifícios (CP7 3800-901, 3800-902 e 3800-903) ficaram a mais de 1 km de qualquer área de serviço modelada e foram excluídos do dimensionamento (Secção 1). O seu impacto agregado é marginal — cerca de 0,9% da potência fotovoltaica teórica total do município — pelo que não compromete as conclusões agregadas, mas deve ser registado como limitação de cobertura da rede de Voronoi modelada.

In [8]:
import pandas as pd

# Carregar os dois outputs
df_balanco = pd.read_csv(r"data/processed/comparacao_final_limpa_cp7.csv")
df_ptd = pd.read_csv(r"data/processed/potencial_com_mapeamento_ptd.csv", sep=";", encoding="utf-8-sig")

# Cruzar pelo CP7
df_merged = pd.merge(df_ptd[["cp7", "ptd_id"]], df_balanco[["cp7", "producao_total_kwh", "consumo_anual_kwh"]], on="cp7", how="inner")

# Agregar por PTD
agg = df_merged.groupby("ptd_id").agg(
    producao_total_kwh=("producao_total_kwh", "sum"),
    consumo_total_kwh=("consumo_anual_kwh", "sum")
).reset_index()

# Candidatos a CER: PTDs onde produção >= consumo
candidatos = agg[agg["producao_total_kwh"] >= agg["consumo_total_kwh"]]
print(f"Candidatos a CER: {len(candidatos)}")

Candidatos a CER: 126
